<a href="https://colab.research.google.com/github/0xfffddd/Coding/blob/main/HW2_DPO_Lichen_Mao_Task_B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 9: LLM Alignment — Direct Preference Optimization (DPO)
### MGMT 590 — Will Wei Sun  
Daniels School of Business, Purdue University

This notebook demonstrates a simple **DPO (Direct Preference Optimization)** workflow for LLM alignment using a small custom preference dataset and Hugging Face **TRL**.

## What this demo shows
- how to format pairwise preference data as `prompt / chosen / rejected`
- how to fine-tune a small instruct model with **DPO**
- how to compare model outputs **before** and **after** preference alignment

## Main references
1. Rafailov et al. (2023), *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*  
2. Hugging Face **TRL** documentation for `DPOTrainer`

## Files needed
Upload both of these files into your Colab session:
1. `custom_preferences_rally_style.jsonl`
2. this notebook


## 1. Install packages

In [1]:

# Run this in a fresh Colab runtime if needed
!pip uninstall -y -q trl transformers peft accelerate datasets tokenizers bitsandbytes
!pip install -q -U "trl[peft]" datasets accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 47.0 MB/s eta 0:00:00


## 2. Imports and version check

In [2]:
import os
import random
import textwrap
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from trl import DPOTrainer, DPOConfig
import transformers, trl, datasets

print("Package versions:")
print(f"  torch        : {torch.__version__}")
print(f"  transformers : {transformers.__version__}")
print(f"  trl          : {trl.__version__}")
print(f"  datasets     : {datasets.__version__}")


Package versions:
  torch        : 2.10.0+cu128
  transformers : 5.5.4
  trl          : 1.1.0
  datasets     : 4.8.4


## 3. Configuration

In [16]:

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
CUSTOM_DATA_FILE = "./lichen-mao-aggressive.jsonl"

BETA = 0.3          #varying it {0.1, 0.5} to see performance difference
LR = 1e-5
EPOCHS = 3          #smaller value speeds up the training
BATCH_SIZE = 1
GRAD_ACCUM = 4
MAX_LENGTH = 384
MAX_PROMPT_LENGTH = 192


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = (
    torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported())
    else torch.float16 if DEVICE == "cuda"
    else torch.float32
)

SEED = 39298551
set_seed(SEED)
random.seed(SEED)

print(f"Running on: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Warning: no GPU detected — training will be slow.")


Running on: cuda
GPU  : Tesla T4
VRAM : 15.6 GB


## 4. Load tokenizer and models

In [17]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading BEFORE-DPO model ...")
before_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)
before_model.eval()

print("Loading trainable model ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)
model.config.use_cache = False

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Trainable model loaded | Parameters: {n_params:.0f}M")


Loading BEFORE-DPO model ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading trainable model ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Trainable model loaded | Parameters: 494M


## 5. Load custom preference dataset

In [18]:

def load_custom_dataset(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".jsonl":
        ds = load_dataset("json", data_files=filepath, split="train")
    elif ext == ".csv":
        ds = load_dataset("csv", data_files=filepath, split="train")
    else:
        raise ValueError("Only .jsonl and .csv are supported.")

    required_cols = {"prompt", "chosen", "rejected"}
    if not required_cols.issubset(set(ds.column_names)):
        raise ValueError(f"Dataset must contain columns {required_cols}, but found {ds.column_names}")
    return ds

train_ds = load_custom_dataset(CUSTOM_DATA_FILE)

print("Custom dataset size:", len(train_ds))
print("\nFirst example:")
print(train_ds[0])
print(train_ds[1])
print(train_ds[2])
print(train_ds[3])
print(train_ds[4])
print(train_ds[5])


Generating train split: 0 examples [00:00, ? examples/s]

Custom dataset size: 50

First example:
{'prompt': "Rewrite this message in a diplomatic decline or refusal style: 'I can't help with this.'", 'chosen': 'Thanks for reaching out—unfortunately I’m not able to help with this right now.', 'rejected': "I can't help with this."}
{'prompt': "Rewrite this message in a diplomatic decline or refusal style: 'No, I won’t do that.'", 'chosen': 'I’m afraid I won’t be able to take this on, but I appreciate you thinking of me.', 'rejected': 'No, I won’t do that.'}
{'prompt': "Rewrite this message in a diplomatic decline or refusal style: 'I’m not interested.'", 'chosen': 'Thanks for sharing this, but I’ll have to pass for now.', 'rejected': 'I’m not interested.'}
{'prompt': "Rewrite this message in a diplomatic decline or refusal style: 'I don’t have time.'", 'chosen': 'I’m a bit tied up at the moment, so I won’t be able to take this on.', 'rejected': 'I don’t have time.'}
{'prompt': "Rewrite this message in a diplomatic decline or refusal style: 'St

## 6. Configure and train DPO

In [19]:

print("Configuring DPO trainer ...")
print(f"  beta={BETA}  |  lr={LR}  |  epochs={EPOCHS}")
print(f"  effective batch = {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")

cfg = DPOConfig(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    beta=BETA,
    max_length=MAX_LENGTH,
    logging_steps=5,
    output_dir="./dpo_out_custom_rally_style",
    report_to="none",
    save_strategy="no",
    eval_strategy="no",
    remove_unused_columns=False,
    fp16=False,
    bf16=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=cfg,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

trainer.train()
print("\nTraining complete!")


Configuring DPO trainer ...
  beta=0.3  |  lr=1e-05  |  epochs=3
  effective batch = 1 x 4 = 4


Adding EOS to train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.158796
10,0.000298
15,0.000004
20,0.000006
25,0.000001
30,0.000001
35,0.000004



Training complete!


## 7. Compare BEFORE vs AFTER

In [20]:
TEST_PROMPTS = [
    "Rewrite this message in a diplomatic decline or refusal style: 'I’m not going to help with this anymore.'",
    "Rewrite this message in a diplomatic decline or refusal style: 'Your request doesn’t make sense, so I won’t do it.'",
    "Rewrite this message in a diplomatic decline or refusal style: 'I don’t want to be part of this project at all.'",
    "Rewrite this message in a diplomatic decline or refusal style: 'I’m rejecting this proposal because it’s a bad idea.'",
    "Rewrite this message in a diplomatic decline or refusal style: 'I won’t answer any more questions about this.'",
]


def generate(model_obj, prompt, max_new_tokens=120):
    model_obj.eval()

    messages = [{"role": "user", "content": prompt}]
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        text = f"User: {prompt}\nAssistant:"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
    ).to(model_obj.device)

    with torch.no_grad():
        out = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Generating responses ...\n")
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print("\n" + "=" * 80)
    print(f"Prompt {i}: {prompt}")
    print("=" * 80)

    before = generate(before_model, prompt)
    after = generate(model, prompt)

    print("\nBEFORE DPO:")
    print(textwrap.indent(before, "    "))

    print("\nAFTER DPO:")
    print(textwrap.indent(after, "    "))


Generating responses ...


Prompt 1: Rewrite this message in a diplomatic decline or refusal style: 'I’m not going to help with this anymore.'

BEFORE DPO:
    "I am unable to assist you further at this time."

AFTER DPO:
    I regret to inform you that I am unable to assist with the matter at this time due to my current commitments and responsibilities.

Prompt 2: Rewrite this message in a diplomatic decline or refusal style: 'Your request doesn’t make sense, so I won’t do it.'

BEFORE DPO:
    I regret to inform you that my current capabilities and resources do not allow me to fulfill your request as requested. Therefore, I am unable to proceed with the task at hand.

AFTER DPO:
    I appreciate your consideration for my request, but I believe there may be an alternative solution that would better suit our needs and align with our goals.

Prompt 3: Rewrite this message in a diplomatic decline or refusal style: 'I don’t want to be part of this project at all.'

BEFORE DPO:
    I regre

## 8. Optional: save the fine-tuned model

In [ ]:

# Uncomment if you want to save the tuned model
# model.save_pretrained("./dpo_out_custom_style/final_model")
# tokenizer.save_pretrained("./dpo_out_custom_style/final_model")
